In [25]:
# !pip install scikeras

# # Uninstall the current scikit-learn version
# !pip uninstall scikit-learn -y

# # Install a compatible version of scikit-learn (e.g., 1.4.2)
# !pip install scikit-learn==1.4.2

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasClassifier

from sklearn.calibration import CalibrationDisplay
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,IterativeImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    multilabel_confusion_matrix,
    f1_score
)

In [27]:
training_df = pd.read_csv(
    filepath_or_buffer='training_faults_diagnostics.csv',
    low_memory=False
)

In [28]:
# Convert SPN and FMI values to strings
training_df['spn'] = training_df['spn'].astype(str)
training_df['fmi'] = training_df['fmi'].astype(str)
print(f'spn datatype: {training_df['spn'].dtype}')

spn datatype: object


In [29]:
target = 'Derate_Target'

# Create dataset with features
X = training_df

# Create array of targets
y = training_df[target]

## Identify features for imputing missing values

In [30]:
# Group categorical columns
categorical_columns = ['EquipmentID', 'spn', 'fmi', 'active', 'Severity_Level']

# Group numeric columns
numeric_columns = [
    'BarometricPressure',
    'EngineCoolantTemperature',
    'EngineLoad',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'Speed',
    'Throttle',
    'TurboBoostPressure'
  ]

## Split training dataset

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

## Create pipeline and fit model

In [32]:
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]
)

numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('numeric_imputer', IterativeImputer(max_iter=20, random_state=30))
    ]
)

In [33]:
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        ('numeric_pipe', numeric_pipe, numeric_columns)
    ]
)

In [34]:
# Get the number of features after the ColumnTransformer's transformation
ct.fit(X_train)
X_val_t = ct.transform(X_val)
X_val_t.shape

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:801: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


(105807, 1412)

In [39]:
n_features = ct.transform(X_train[:1]).shape[1]

es = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Function to create the Keras model for SciKeras
def create_model():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(shape=(n_features,)))
    model.add(tf.keras.layers.Dense(64, activation='relu'))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(3, activation='softmax'))
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=[
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )
    return model

# Keras model with SciKeras wrapper
model = KerasClassifier(
    model=create_model,
    callbacks=[es],
    epochs=20,
    batch_size=32
)


In [40]:
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', model)
    ]
)

In [42]:
pipe.fit(
    X=X_train,
    y=tf.keras.utils.to_categorical(
        y_train.values,
        num_classes=3
    ),  # One-hot encode y_train
    model__validation_data=(
        X_val_t,
        tf.keras.utils.to_categorical(
            y_val.values,
            num_classes=3
        )
    ) # One-hot encode y_val
)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:801: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Epoch 1/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 75s 2ms/step - loss: 0.0092 - precision_3: 0.9985 - recall_3: 0.9976 - val_loss: 0.0075 - val_precision_3: 0.9986 - val_recall_3: 0.9985
Epoch 2/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 69s 2ms/step - loss: 0.0072 - precision_3: 0.9986 - recall_3: 0.9985 - val_loss: 0.0073 - val_precision_3: 0.9986 - val_recall_3: 0.9986
Epoch 3/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 69s 2ms/step - loss: 0.0070 - precision_3: 0.9986 - recall_3: 0.9985 - val_loss: 0.0071 - val_precision_3: 0.9986 - val_recall_3: 0.9985
Epoch 4/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 69s 2ms/step - loss: 0.0068 - precision_3: 0.9986 - recall_3: 0.9985 - val_loss: 0.0068 - val_precision_3: 0.9986 - val_recall_3: 0.9985
Epoch 5/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 69s 2ms/step - loss: 0.0066 - precision_3: 0.9987 - recall_3: 0.9985 - val_loss: 0.0070 - val_precision_3: 0.9986 - val_recall_3: 0.9984
Epoch 6/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 69s 2ms/step - loss: 0.0066 - precision_3: 0.9987 -

Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('categorical_pipe',
                                                  Pipeline(steps=[('categorical_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EquipmentID', 'spn', 'fmi',
                                                   'active',
                                                   'Severity_Level']),
                                                 ('numeric_pipe',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('numeric_imputer',
                                                                   Iter...
                                                   'EngineLoad',
                                                   'EngineOilPressure',
                                                   'EngineOilTemperature',
                                                   'EngineRpm', 'FuelRate',
                                                   'FuelTemperature',
                                                   'IntakeManifoldTemperature',
                                                   'Speed', 'Throttle',
                                                   'TurboBoostPressure'])])),
                ('model',
                 KerasClassifier(batch_size=32, callbacks=[<keras.src.callbacks.early_stopping.EarlyStopping object at 0x7d7725c56ba0>], epochs=20, model=<function create_model at 0x7d7a9b93d440>))])

## Compare Training and Testing

In [43]:
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

29759/29759 ━━━━━━━━━━━━━━━━━━━━ 72s 2ms/step
9920/9920 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step


In [48]:
# One-hot encode y_train for classification_report and multilabel_confusion_matrix
y_train_ohe = tf.keras.utils.to_categorical(y_train.values, num_classes=3)

# Convert one-hot encoded predictions to integer labels for confusion_matrix
y_pred_train_int = np.argmax(y_pred_train, axis=1)

training_cr = classification_report(
    y_true=y_train_ohe,
    y_pred=y_pred_train,
    digits=6
)
print(str(training_cr))

training_cm = confusion_matrix(
    y_true=y_train.values, # Use original integer labels for y_true
    y_pred=y_pred_train_int # Use integer predictions for y_pred
)
print(training_cm)

              precision    recall  f1-score   support

           0   0.998783  0.999901  0.999342    950562
           1   0.642857  0.011264  0.022140       799
           2   0.725080  0.500555  0.592252       901

   micro avg   0.998599  0.998599  0.998599    952262
   macro avg   0.788907  0.503907  0.537911    952262
weighted avg   0.998226  0.998599  0.998137    952262
 samples avg   0.998599  0.998599  0.998599    952262

[[950468      2     92]
 [   711      9     79]
 [   447      3    451]]


In [49]:
# One-hot encode y_test for classification_report and multilabel_confusion_matrix
y_test_ohe = tf.keras.utils.to_categorical(y_test.values, num_classes=3)

# Convert one-hot encoded predictions to integer labels for confusion_matrix
y_pred_test_int = np.argmax(y_pred_test, axis=1)

testing_cr = classification_report(
    y_true=y_test_ohe,
    y_pred=y_pred_test,
    digits=6
)
print(str(testing_cr))

testing_cm = confusion_matrix(
    y_true=y_test.values, # Use original integer labels for y_true
    y_pred=y_pred_test_int # Use integer predictions for y_pred
)
print(testing_cm)

              precision    recall  f1-score   support

           0   0.998764  0.999912  0.999338    316854
           1   0.750000  0.011236  0.022140       267
           2   0.708543  0.470000  0.565130       300

   micro avg   0.998579  0.998579  0.998579    317421
   macro avg   0.819102  0.493716  0.528869    317421
weighted avg   0.998281  0.998579  0.998105    317421
 samples avg   0.998579  0.998579  0.998579    317421

[[316826      0     28]
 [   234      3     30]
 [   158      1    141]]
